# Lab 10: check Gemini's labels, then use them

**50–60 minutes.** Work with your team. Make a copy named `lab10_mine.ipynb`. Switch who types halfway through.

The question in this lab: does the poster of an r/AmItheAsshole post admit fault? No column in the data answers it. You label a few posts yourselves, check Gemini against hand labels, and then use Gemini on posts it has not seen.

You need your Gemini key in the `.env` file. The lab makes 70 calls, and 30 more if you do the last part.

In [ ]:
from pathlib import Path
import json
import logging
import os
import time

import pandas as pd
from sklearn.metrics import cohen_kappa_score
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"

logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call
pd.set_option("display.max_colwidth", 300)  # show more of each text
print("Key loaded.")

## 1. Load the hand-labeled posts and set a baseline

`week10_post_labels.csv` has 30 posts, 15 NTA and 15 YTA. The `hand` column is a hand label for this definition:

- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

In [ ]:
labels = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_post_labels.csv")
labels["hand"].value_counts()  # Check: no 26, yes 4

In [ ]:
# Your turn: the accuracy of always guessing "no". Check: 0.87

## 2. Make your own answer key

Hand labels are one person's reading of the definition. Before you use them to check a model, check whether two people agree.

Two of you: read the first 8 posts on your own and label each one yes or no with the definition above. Don't talk, and don't look at the other columns, until you are both done.

In [ ]:
to_label = labels.head(8)
to_label[["title", "text"]]

In [ ]:
# Type your labels in the order of the table, for example ["no", "yes", "no", ...]
person_a = []
person_b = []
len(person_a), len(person_b)  # Check: 8 and 8

In [ ]:
# Example: agreement, the share of posts the two of you gave the same label
ab = pd.DataFrame({"a": person_a, "b": person_b, "hand": to_label["hand"].values})
(ab["a"] == ab["b"]).mean()

In [ ]:
# Cohen's kappa: agreement corrected for the agreement you would get by chance. 1 is perfect agreement, 0 is what chance gives
cohen_kappa_score(ab["a"], ab["b"])

In [ ]:
# Your turn: the agreement of each of you with hand

Find a post where the two of you disagree, or where you disagree with `hand`. Is the definition unclear for that post? Write one sentence you could add to the definition to settle it.

Your answer:

## 3. Check Gemini on the 30 posts

Gemini gets the same definition as its instruction. Read the cells, then run them.

In [ ]:
SYSTEM = """You read posts from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Label whether the poster admits that they did something wrong.
- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

Give a reason of one sentence."""

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema={
        "type": "object",
        "properties": {
            "llm_label": {"type": "string", "enum": ["yes", "no"]},
            "llm_reason": {"type": "string"},
        },
        "required": ["llm_label", "llm_reason"],
    },
)

In [ ]:
# Send one prompt, and try again after error 429 or 503.
def ask(prompt, config, max_tries=4):
    wait = 10
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in {429, 503} or attempt == max_tries:
                raise
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2

In [ ]:
# Label every post in a table and return the labels next to the table.
# Each answer is saved to the file right away. If a cell that uses this stops, run it again: posts already in the file are skipped.
def label_all(table, path, config):
    path = Path(path)
    done = set(pd.read_csv(path)["id"]) if path.exists() else set()
    for row in table.itertuples():
        if row.id in done:
            continue
        answer = ask(f"Title: {row.title}\n\n{row.text}", config)
        line = pd.DataFrame([{"id": row.id, **answer, "model": GEMINI_MODEL}])
        line.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(row.id)
        print(len(done), "of", len(table), "done")
    return pd.read_csv(path).merge(table, on="id")

In [ ]:
gemini = label_all(labels, "lab10_gemini_30.csv", CONFIG)  # takes about 4 minutes
len(gemini)  # Check: 30

In [ ]:
# Your turn: the accuracy of Gemini's labels (llm_label) against hand. Is it better than always guessing "no"?
# Check: about 0.8 to 0.9. Gemini's number can change between runs

In [ ]:
# Your turn: a confusion table with hand in the rows and llm_label in the columns. Hint: pd.crosstab(rows, columns)
# Then count the false positives (hand no, Gemini yes) and the false negatives (hand yes, Gemini no)

In [ ]:
# The file's gemini column is an earlier run with the same instruction. How many of your labels differ from it?
(gemini["llm_label"] != gemini["gemini"]).sum()  # Check: usually 0 to 3

In [ ]:
# The posts where Gemini and hand disagree, with Gemini's reason
gemini.loc[gemini["llm_label"] != gemini["hand"], ["title", "hand", "llm_label", "llm_reason"]]

Does Gemini make more false positives or more false negatives? Read one post of the more common kind (the full text is in the `text` column). Why do you think Gemini labeled it that way?

Your answer:

## 4. Use Gemini on posts it has not seen

`week10_posts.csv` has 785 posts with a verdict but no hand label. Gemini labels 20 NTA and 20 YTA posts that are not among the 30. Then you compare how often the posters admit fault in each group.

In [ ]:
posts = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_posts.csv")
unseen = posts[~posts["id"].isin(labels["id"])]  # leave out the 30 hand-labeled posts
sample = unseen.groupby("verdict").sample(n=20, random_state=211)
sample["verdict"].value_counts()  # Check: NTA 20, YTA 20

In [ ]:
new = label_all(sample, "lab10_gemini_new.csv", CONFIG)  # takes about 5 minutes
len(new)  # Check: 40

In [ ]:
# Your turn: the share of NTA posts and of YTA posts that Gemini labels yes.
# Hint: pd.crosstab(new["verdict"], new["llm_label"], normalize="index") gives shares in each row

With 20 posts in each group, one post is 5 percentage points. Think of what you found in part 3: does Gemini say yes too often or too rarely? Can you say that YTA posters admit fault more often than NTA posters? What would you check first?

Your answer:

## 5. Describe what you found

Write three or four sentences. Say how well Gemini did on the 30 posts, compared with always guessing "no", and how well the two of you agreed with each other. Then say whether you would use Gemini's labels to compare NTA and YTA posts, and what you would check first.

Your finding:

Check that your key does not appear in any cell or output. Then save the notebook.

## If you have time: add a rule and run it again

Pick one post Gemini got wrong in part 3. Add one rule to the definition that would settle it, for example "Asking whether you were wrong is not admitting it." Label the 30 posts again into a new file and compare the two runs.

In [ ]:
# A new SYSTEM2 and CONFIG2 with your rule

In [ ]:
# Label the 30 posts again with CONFIG2 into lab10_gemini_30_v2.csv

In [ ]:
# How many labels changed between the two runs? What is the new accuracy against hand?

You wrote the rule after reading these 30 posts and scored it on the same 30. Why is the new accuracy probably too high? What would you need to score it fairly?

Your answer: